# DER Inventory

Connect Critical Facilities to the synthetic electrical network with DER rows (backup gen, power source)

> **Stage Contract**
>
> Requires: base network asset registry, Stage A1 `assets.parquet` / `control_units.parquet`, and location-local `critical_facilities.geojson`
>
> Produces: reviewed Critical Facilities, Load Matches, and Layer 1 DER seed rows
>
> Next: `02_load_profiles.ipynb`


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from study_location import bootstrap
session = bootstrap()
location_root = session.location_root
repo_root = session.repo_root

# load Location Workspace config and standard grid paths.
from location_runtime import build_grid_paths
from location_runtime import load_runtime as load_config_runtime

config, paths = load_config_runtime(location_root / "config.yaml")
grid = build_grid_paths(config)
location_root = paths["location_root"]
location_name = paths["location_name"]
repo_root = paths["repo_root"]
for key in (
    "shift_cache",
    "opendss_root",
    "asset_registry",
    "augmented_artifacts",
    "onm_export",
    "figures",
):
    grid[key].mkdir(parents=True, exist_ok=True)

augmented_artifacts: /home/grahamhults/projects/Flood-RM/locations/marshfield/data/static/power_grid/smart_ds_compat


## Layer 1 — Evidence-Anchored DER Seeds

Get location-specific critical facility data, match loads to critical facilities, seed DER rows for backup-power to facilities, then validate and write the artifacts.

A *seed row* is a starter record — not a random-number seed. Each one marks **which** Critical Facility has backup power and **where** it attaches to the grid, while leaving **how much** (capacity) blank.

In [ ]:
from power.resilience import (
    build_der_inventory,
    build_load_matches,
    der_version,
    facility_version,
    load_bus_electrical_metadata,
    load_critical_facilities,
    load_match_version,
    validate_der_assignments,
    validate_load_matches,
    write_critical_facilities_artifact,
    write_der_inventory,
    write_load_matches,
)

The method starts from Stage A1 grid artifacts (SHIFT) and the reviewed location-local Critical Facility evidence file.

In [ ]:
assets_path = grid["augmented_artifacts"] / "assets.parquet"
control_units_path = grid["augmented_artifacts"] / "control_units.parquet"
critical_facilities_source = location_root / config["grid"]["critical_facilities"]["source"]
critical_facilities_path = grid["augmented_artifacts"] / "critical_facilities.parquet"
load_matches_path = grid["augmented_artifacts"] / "critical_load_assignments.parquet"
der_inventory_path = grid["augmented_artifacts"] / "der_inventory.parquet"

if not assets_path.exists() or not control_units_path.exists():
    missing = [p.relative_to(repo_root) for p in (assets_path, control_units_path) if not p.exists()]
    raise FileNotFoundError(
        "Run 01_base_network.ipynb first; missing Stage A1 artifacts: "
        + ", ".join(str(p) for p in missing)
    )


Load the reviewed facility source and write the canonical Critical Facility artifact used by later Augmented Network steps.


In [ ]:
assets = pd.read_parquet(assets_path)
control_units = pd.read_parquet(control_units_path)
loads = pd.read_csv(grid["asset_registry"] / "loads.csv")
buses = pd.read_csv(grid["asset_registry"] / "buses.csv")

critical_facility_gdf = load_critical_facilities(
    critical_facilities_source,
    location_name=paths["location_name"],
)
write_critical_facilities_artifact(critical_facility_gdf, critical_facilities_path)
critical_facility_records = critical_facility_gdf.drop(columns=["geometry"], errors="ignore").to_dict("records")
print(f"Critical Facilities: {len(critical_facility_records)} ({facility_version})")



Each public Critical Facility is matched to a synthetic load-bus service proxy with explicit provenance; this is not a utility service-record claim.

In [ ]:
load_match_rows = build_load_matches(
    critical_facility_records,
    asset_rows=assets.to_dict("records"),
    control_unit_rows=control_units.to_dict("records"),
    load_bus_electrical_metadata=load_bus_electrical_metadata(loads),
    location_id=paths["location_name"],
    assign_nearest_when_outside_radius=True,
)
validate_load_matches(
    load_match_rows,
    facility_ids={row["facility_id"] for row in critical_facility_records},
    asset_ids=set(assets["asset_id"].dropna().astype(str)),
    control_unit_ids=set(control_units["control_unit_id"].dropna().astype(str)),
)
write_load_matches(load_match_rows, load_matches_path)
print(f"Load Matches: {len(load_match_rows)} ({load_match_version})")


Layer 1 adds one DER seed row for each facility with documented or planned-authorized backup power. Capacity fields stay null until Layer 2 sizing.

In [ ]:
layer1_rows = build_der_inventory(
    critical_facility_records,
    load_matches=load_match_rows,
    location_id=paths["location_name"],
)
validate_der_assignments(
    layer1_rows,
    valid_buses=set(buses["bus"].dropna().astype(str)),
)
write_der_inventory(layer1_rows, der_inventory_path)
der_inventory = pd.DataFrame(layer1_rows)
print(f"Layer 1 DER rows: {len(layer1_rows)} ({der_version})")


Review the facility-to-grid attachment fields before moving to load profiles and REopt sizing.

In [ ]:
display(der_inventory[["facility_id", "load_asset_id", "bus", "assignment_status", "placement_rule", "genset_kw"]].head(20))

### Layer 2 — REopt Sizing

Layer 2 sizing depends on `load_profile_assignments.parquet`, which is built in the next chronological notebook.

In [ ]:
from power.resilience import CachedReoptResultsClient, size_der

print("Layer 2 REopt replay is intentionally deferred until after 02_load_profiles.ipynb.")
print("Use size_der(...) with a live or cached client when refreshing capacities.")

### Write Artifact

In [ ]:
print(f"Wrote {len(der_inventory):,} rows to {der_inventory_path.relative_to(repo_root)}")
print(f"Critical Facilities: {critical_facilities_path.relative_to(repo_root)}")
print(f"Load Matches: {load_matches_path.relative_to(repo_root)}")